# 03 — Anàlisi de la baseline real

Calcula els km i temps de la ruta **original** (com va fer-ho el repartidor el 5/2/2026).
Aquest és el punt de comparació per al pitch: original vs. proposta optimitzada.

**Output**: `data/processed/baseline_real.json`

In [1]:
import pandas as pd
import numpy as np
import json
from pathlib import Path
from math import radians, sin, cos, sqrt, atan2

PROCESSED = Path('../data/processed')
RAW = Path('../data/raw')

# Carregar clients geocodificats
clients_geo = pd.read_csv(PROCESSED / 'clients_geo.csv')
print(f'Clients carregats: {len(clients_geo)} (inclou depot)')
print(clients_geo[['nom', 'poblacio', 'palets']].to_string())

Clients carregats: 26 (inclou depot)
                       nom                    poblacio  palets
0       DDI Mollet (Depot)           Mollet del Vallès    0.00
1      RESTAURANT EL ROSER                 CALLDETENES    0.30
2       CELLER CALLDETENES                 CALLDETENES    0.10
3                   SUKIPA                 CALLDETENES    0.40
4      BAR DE LA BENZINERA                 CALLDETENES    0.50
5               CA LA NENA                 CALLDETENES    0.30
6               BAR KARNAK                 FOLGUEROLES    0.30
7       L'ESPAI RESTAURANT                 FOLGUEROLES    0.35
8   LA COCA DE FOLGUEROLES                 FOLGUEROLES    0.15
9            CAL CISTELLER                 FOLGUEROLES    0.25
10    BAR PAVELLO ST JULIA     SANT JULIÀ DE VILATORTA    0.20
11             BAR EL TUPÍ     SANT JULIÀ DE VILATORTA    0.40
12          MAS L'ALBAREDA     SANT JULIÀ DE VILATORTA    0.25
13      BAR NURIA ST.JULIA     SANT JULIÀ DE VILATORTA    0.90
14            CAL 

In [2]:
# Funció per calcular distància entre dos punts (Haversine)
# Retorna km
def haversine(lat1, lon1, lat2, lon2):
    R = 6371  # radi de la Terra en km
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1-a))

# Matriu de distàncies entre tots els punts
n = len(clients_geo)
dist_matrix = np.zeros((n, n))

for i in range(n):
    for j in range(n):
        if i != j:
            dist_matrix[i][j] = haversine(
                clients_geo.iloc[i]['lat'], clients_geo.iloc[i]['lon'],
                clients_geo.iloc[j]['lat'], clients_geo.iloc[j]['lon']
            )

print(f'Matriu de distàncies: {n}x{n}')
print(f'Distància Mollet → Vic centre: {dist_matrix[0][18]:.1f} km')
print(f'Distància Mollet → Calldetenes: {dist_matrix[0][1]:.1f} km')
print(f'Distància Mollet → St Julià: {dist_matrix[0][10]:.1f} km')

Matriu de distàncies: 26x26
Distància Mollet → Vic centre: 43.1 km
Distància Mollet → Calldetenes: 41.1 km
Distància Mollet → St Julià: 45.5 km


In [3]:
# Ordre original de la ruta (basat en les dades reals del transport 11443257)
# Hem inferit l'ordre a partir de l'ordre d'aparició al dataset
# i la lògica geogràfica de la zona Vic
#
# Nota: sense GPS tracker real, usem l'ordre d'aparició al dataset
# com a proxy de l'ordre de visita original del repartidor

# Ordre original (índexs al dataframe clients_geo, 0=depot)
# Ruta típica d'un repartidor de la zona: surt de Mollet, puja per
# l'AP-7, primer para a Calldetenes, segueix a Folgueroles,
# puja a St Julià de Vilatorta, baixa a Vic, torna a Mollet

ordre_original = [
    0,   # DDI Mollet (Depot) - sortida
    1,   # RESTAURANT EL ROSER (Calldetenes)
    2,   # CELLER CALLDETENES
    3,   # SUKIPA
    4,   # BAR DE LA BENZINERA
    5,   # CA LA NENA
    6,   # BAR KARNAK (Folgueroles)
    7,   # L'ESPAI RESTAURANT
    8,   # LA COCA DE FOLGUEROLES
    9,   # CAL CISTELLER
    10,  # BAR PAVELLO ST JULIA (St Julià de Vilatorta)
    11,  # BAR EL TUPÍ
    12,  # MAS L'ALBAREDA
    13,  # BAR NURIA ST.JULIA
    14,  # CAL TEIXIDOR
    15,  # MGA GP VIC (Vic)
    16,  # GUSTUM AREAS 1
    17,  # GUSTUM AREAS 2
    18,  # SNACK (VIC)
    19,  # BAR LA PISTA
    20,  # EL CALIU RESTAURANT
    21,  # VINS I BEGUDES REMEI
    22,  # BAR SOL NOU
    23,  # BAR PRIEGO
    24,  # LA TAVERNA D'EN NORTON
    25,  # 412 COFFEE BEER (Sant Fost) - desvio de tornada
    0    # DDI Mollet (Depot) - retorn
]

# Calcular km totals de la ruta original
km_original = 0
for i in range(len(ordre_original)-1):
    origen = ordre_original[i]
    desti = ordre_original[i+1]
    km_tram = dist_matrix[origen][desti]
    km_original += km_tram

# Factor de correcció: la distància Haversine és en línia recta
# Per carretera, multipliquem per ~1.3 (factor estàndard per zones rurals)
FACTOR_CARRETERA = 1.30
km_original_carretera = km_original * FACTOR_CARRETERA

print(f'Km línia recta: {km_original:.1f} km')
print(f'Km estimats per carretera (x{FACTOR_CARRETERA}): {km_original_carretera:.1f} km')

# Desglossat per trams
print('\nDesglossat per trams:')
for i in range(len(ordre_original)-1):
    o = ordre_original[i]
    d = ordre_original[i+1]
    km = dist_matrix[o][d] * FACTOR_CARRETERA
    nom_o = clients_geo.iloc[o]['nom'][:25]
    nom_d = clients_geo.iloc[d]['nom'][:25]
    print(f'  {nom_o:<25} → {nom_d:<25} {km:5.1f} km')

Km línia recta: 109.5 km
Km estimats per carretera (x1.3): 142.4 km

Desglossat per trams:
  DDI Mollet (Depot)        → RESTAURANT EL ROSER        53.4 km
  RESTAURANT EL ROSER       → CELLER CALLDETENES          0.1 km
  CELLER CALLDETENES        → SUKIPA                      0.1 km
  SUKIPA                    → BAR DE LA BENZINERA         0.2 km
  BAR DE LA BENZINERA       → CA LA NENA                  0.1 km
  CA LA NENA                → BAR KARNAK                  3.6 km
  BAR KARNAK                → L'ESPAI RESTAURANT          0.1 km
  L'ESPAI RESTAURANT        → LA COCA DE FOLGUEROLES      0.1 km
  LA COCA DE FOLGUEROLES    → CAL CISTELLER               0.1 km
  CAL CISTELLER             → BAR PAVELLO ST JULIA        3.2 km
  BAR PAVELLO ST JULIA      → BAR EL TUPÍ                 0.1 km
  BAR EL TUPÍ               → MAS L'ALBAREDA              0.2 km
  MAS L'ALBAREDA            → BAR NURIA ST.JULIA          0.3 km
  BAR NURIA ST.JULIA        → CAL TEIXIDOR                0.3 km

In [4]:
# Estimació del temps total de la ruta original
# Velocitat mitjana per carretera comarcal/poble: ~40 km/h
# Temps de descàrrega per client: 15 min de mitjana (varia molt)
# Número de clients: 25

VELOCITAT_KMH = 40
TEMPS_DESCÀRREGA_MIN = 15  # minuts per client
N_CLIENTS = 25

temps_conduccio_h = km_original_carretera / VELOCITAT_KMH
temps_descàrrega_h = (TEMPS_DESCÀRREGA_MIN * N_CLIENTS) / 60
temps_total_h = temps_conduccio_h + temps_descàrrega_h

print(f'=== BASELINE ORIGINAL ===')
print(f'Km totals (estimats): {km_original_carretera:.1f} km')
print(f'Temps conducció: {temps_conduccio_h:.1f} h ({temps_conduccio_h*60:.0f} min)')
print(f'Temps descàrrega ({N_CLIENTS} clients x {TEMPS_DESCÀRREGA_MIN} min): {temps_descàrrega_h:.1f} h')
print(f'TEMPS TOTAL ESTIMAT: {temps_total_h:.1f} h ({temps_total_h*60:.0f} min)')
print()
print(f'Clients visitats: {N_CLIENTS}')
print(f'Palets totals: {clients_geo[clients_geo["client_id"]!=0]["palets"].sum():.1f}')

=== BASELINE ORIGINAL ===
Km totals (estimats): 142.4 km
Temps conducció: 3.6 h (214 min)
Temps descàrrega (25 clients x 15 min): 6.2 h
TEMPS TOTAL ESTIMAT: 9.8 h (589 min)

Clients visitats: 25
Palets totals: 11.8


In [5]:
# Guardar baseline com a JSON per al pitch
baseline = {
    'transport_id': 11443257,
    'data': '2026-02-05',
    'ruta': 'DR0051',
    'repartidor': 850013,
    'n_clients': N_CLIENTS,
    'n_linies_comanda': 196,
    'palets_totals': round(float(clients_geo[clients_geo['client_id']!=0]['palets'].sum()), 2),
    'km_linia_recta': round(km_original, 1),
    'km_carretera_estimats': round(km_original_carretera, 1),
    'factor_carretera': FACTOR_CARRETERA,
    'temps_conduccio_h': round(temps_conduccio_h, 2),
    'temps_descàrrega_h': round(temps_descàrrega_h, 2),
    'temps_total_h': round(temps_total_h, 2),
    'velocitat_assumida_kmh': VELOCITAT_KMH,
    'temps_descàrrega_per_client_min': TEMPS_DESCÀRREGA_MIN,
    'ordre_original': ordre_original,
    'supòsits': [
        'Ordre inferit de les dades reals del dataset (ordre aparició)',
        f'Factor carretera x{FACTOR_CARRETERA} (estàndard zones rurals)',
        f'Velocitat mitjana {VELOCITAT_KMH} km/h (carreteres comarcals)',
        f'{TEMPS_DESCÀRREGA_MIN} min de descàrrega per client (mitjana estimada)',
        'Distància calculada amb fórmula Haversine (línia recta entre coordenades)'
    ]
}

OUT = PROCESSED / 'baseline_real.json'
with open(OUT, 'w', encoding='utf-8') as f:
    json.dump(baseline, f, indent=2, ensure_ascii=False)

print(f'✓ Guardat a {OUT}')
print()
print('=== RESUM PER AL PITCH ===')
print(f'Ruta original DR0051 — {N_CLIENTS} clients — 5/2/2026')
print(f'  Km estimats:    {km_original_carretera:.0f} km')
print(f'  Temps total:    {temps_total_h:.1f} hores')
print(f'  Palets:         {baseline["palets_totals"]} palets equivalents')
print()
print('Aquests son els numeros que el VRP ha de superar.')
print('baseline_real.json llest per a P4 (pitch) ✓')

✓ Guardat a ..\data\processed\baseline_real.json

=== RESUM PER AL PITCH ===
Ruta original DR0051 — 25 clients — 5/2/2026
  Km estimats:    142 km
  Temps total:    9.8 hores
  Palets:         11.75 palets equivalents

Aquests son els numeros que el VRP ha de superar.
baseline_real.json llest per a P4 (pitch) ✓


## ✅ Feina de P1 completada

Fitxers generats:
- `data/processed/cas_us_clients.csv` — 24 clients amb comandes
- `data/processed/cas_us_linies.csv` — 196 línies de comanda detallades
- `data/processed/clients_geo.csv` — 25 punts (depot + clients) amb lat/lon
- `data/processed/baseline_real.json` — km i temps de la ruta original

**Següent pas per al equip:**
- **P2**: llegir `clients_geo.csv` i fer el VRP amb OR-Tools
- **P3**: llegir `cas_us_linies.csv` i fer la lògica de càrrega del camió
- **P4**: llegir `baseline_real.json` per als KPIs del pitch